# XRD phase identification — v5 (mixtures, any material)

**New in v5**
- **Mixtures**: name the chemical systems you expect (`SYSTEMS`); a stage counts as good enough when the identified phases *together* explain the pattern, not only one phase.
- **Lab Kα1/Kα2 doublet** (`WAVELENGTH = "CuKa"`): every reflection is matched as two peaks, so resolved high-angle doublets are no longer "unexplained".
- **More sensitive peak detection** (noise-based): minor phases near 1 wt% become visible in good data.
- **Lattice parameters for every identified phase** — cubic, tetragonal, hexagonal/trigonal, orthorhombic — using only each phase's own peaks; minor phases reuse the main phase's 2θ zero offset.

Preset: IUCr CPD round-robin sample 1a (corundum + fluorite + zincite). For NMC samples see the comments in the settings cell. All code is in `xrd_refs.py`; **restart the kernel after replacing it**.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
sys.path.insert(0, str(Path.cwd()))
import xrd_refs as X

# ---------------- settings ----------------
LIBRARY = Path(r"reference_library")          # ONE folder for all projects

EXPERIMENTAL_FILE = r"examples/cpd-1a.txt"
OUT        = Path("results/cpd-1a")

ELEMENTS   = ["Al", "Ca", "F", "Zn", "O"]     # all elements in the sample
SYSTEMS    = [["Al", "O"], ["Ca", "F"], ["Zn", "O"]]   # phases' chemical systems you expect; None if unknown
WAVELENGTH = "CuKa"      # "CuKa": lab Cu tube, Ka1+Ka2 doublet (CPD data)   "CuKa1": Ka1 only / synchrotron converted
SIM_RANGE  = (5, 150)

GOOD_SCORE, GOOD_EXPLAINED_PCT, MAX_CELL_SHIFT_PCT = 0.85, 80.0, 1.0
MIXTURE    = True        # judge a stage by all identified phases together
USE_GENERATED_MODELS = False                  # True: add built NMC/LiCoO2 models (battery samples)
IMPURITY_LIBRARY     = []                     # extra side-phase COD searches: a name from X.IMPURITY_LIBRARIES,
                                              # a list like [["Ca","O"], ["Zn","Al","O"]], or [] for none
# NMC example: ELEMENTS = ["Li","Ni","Mn","Co","O"], SYSTEMS = None, WAVELENGTH = "CuKa1" (EVA file),
#              USE_GENERATED_MODELS = True, IMPURITY_LIBRARY = "Li-ion layered oxide (NMC/NCA/LCO)"

MP_API_KEY = "".strip()
if not MP_API_KEY:
    MP_API_KEY = os.environ.get("MP_API_KEY", "")
print("Materials Project key:", f"set ({len(MP_API_KEY)} characters)" if MP_API_KEY else "NOT set (COD only)")
OUT.mkdir(parents=True, exist_ok=True)

## 0. (Optional) add earlier downloads to the library
The repository ships a starter library in `reference_library/`; every new download is added to it automatically.

In [ ]:
# Optional: copy CIFs from older result folders into the library (folders that do not exist are skipped)
# X.add_to_library(["path/to/old_results"], LIBRARY)

## 1. Find references: library first, databases only if needed
With `SYSTEMS` given, any expected system missing from the library is fetched first (one COD query each), then the library is judged. The log shows each stage and why it stopped; for a mixture it also lists the phases and how much of the pattern they explain together.

In [ ]:
built_refs = []
if USE_GENERATED_MODELS:
    built_refs = [X.build_layered_preset(n) for n in ["LiCoO2", "NMC111", "NMC532", "NMC622", "NMC811"]]

x, y = X.load_xy(EXPERIMENTAL_FILE)
print(f"Scan: {len(x)} points, {x.min():.2f}-{x.max():.2f} deg, step {np.median(np.diff(x)):.3f}\n")

all_refs, ranking, search_log = X.auto_reference_search(
    ELEMENTS, x, y, LIBRARY, WAVELENGTH, mp_api_key=MP_API_KEY, extra_refs=built_refs,
    systems=SYSTEMS, mixture=MIXTURE,
    min_score=GOOD_SCORE, min_explained_pct=GOOD_EXPLAINED_PCT, max_cell_shift_pct=MAX_CELL_SHIFT_PCT)

search_log.to_csv(OUT / "search_log.csv", index=False)
ranking.to_csv(OUT / "ranking_vs_experiment.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(search_log)
display(ranking[["source", "ref_id", "formula", "spacegroup", "score", "cell_scale_pct",
                 "explained_pct", "unexplained_peaks", "missing_strong"]].head(12))

## 2. Summary of the best-matching references

In [ ]:
by_id = {r["ref_id"]: r for r in all_refs}
table = X.summary_table([by_id[i] for i in ranking.ref_id.head(60)], SIM_RANGE, WAVELENGTH)
table.to_csv(OUT / "reference_summary.csv", index=False)
display(table[[c for c in ["source", "ref_id", "formula", "spacegroup", "a", "c", "ordered", "group",
                           "strongest_peaks"] if c in table.columns]].head(15))

## 3. Which phases are in the sample?
Cullity's iterative procedure: main phase, then phases explaining the leftover peaks. A minor phase is accepted when it explains at least 3 leftover peaks (its pattern score stays low because only its strongest lines rise above the noise). Isolated single peaks are listed, never assigned.

In [ ]:
if IMPURITY_LIBRARY:
    impurity_refs = X.search_impurities(IMPURITY_LIBRARY, out_dir=LIBRARY)
else:
    impurity_refs = []
phases, report = X.identify_phases(all_refs, x, y, WAVELENGTH, impurity_candidates=impurity_refs)
report.to_csv(OUT / "phase_identification.csv", index=False)
pd.set_option("display.max_colwidth", 200)
display(report)

## 4. Lattice parameters of every identified phase
The main phase is fitted with a free 2θ zero offset; the other phases reuse that offset (it is the same for the whole scan, and a minor phase has too few peaks to fit both). Only peaks that no other identified phase could explain are used. Monoclinic/triclinic phases are skipped.

In [ ]:
cells, zero = [], None
for i, p in enumerate(phases):
    try:
        res, used = X.fit_cell(x, y, p, WAVELENGTH, other_phases=phases, zero_shift=zero)
    except Exception as ex:
        print(f"{p['formula']} ({p['ref_id']}): not fitted - {ex}")
        continue
    if i == 0:
        zero = res["zero_shift_deg"]
    cells.append(res)
    print(f"{p['formula']} {res['system']}: a = {res['a']:.4f} A" + (f", c = {res['c']:.4f} A" if res['system'] != 'cubic' else "")
          + f"  ({res['n_peaks']} peaks, rms {res['rms_deg']:.4f} deg, zero {res['zero_shift_deg']:+.4f} deg"
          + (" fixed" if res["zero_fixed"] else "") + ")" + (f"  WARNING: {res['warning']}" if "warning" in res else ""))
cells = pd.DataFrame(cells)
cells.to_csv(OUT / "lattice_parameters.csv", index=False)
display(cells.round(5))

## 5. Plot: your scan with the identified phases

In [ ]:
by_id.update({p["ref_id"]: p for p in phases})
pick = [p["ref_id"] for p in phases]
for rid in X.pick_for_plot(ranking, n_top=2, per_source=1, table=table):
    if rid not in pick and len(pick) < 6:
        pick.append(rid)
roles = dict(zip(report.ref_id, report.role))
score = dict(zip(ranking.ref_id, ranking.score))
for rid in pick:
    r = by_id[rid]
    r["plot_label"] = f"{r['formula']} [{r['spacegroup']}] {rid} · {roles.get(rid, 'score %.2f' % score.get(rid, float('nan')))}"
peaks = X.find_experimental_peaks(x, X.subtract_background(x, y)[0])
fwhm = float(np.clip(peaks.nlargest(5, "rel_intensity").fwhm.median(), 0.02, 0.3))
plot_range = (float(np.floor(x.min())), float(np.ceil(x.max())))
ax = X.plot_stack([by_id[i] for i in pick], plot_range, WAVELENGTH, fwhm=fwhm, experimental=(x, y))
ax.set_title("Identified phases")
plt.tight_layout(); plt.savefig(OUT / "phases.png", dpi=200); plt.show()

# zoom: intensity axis cut low so ~1 % peaks show; each phase's reflections as a row of ticks below the data
fig, ax = plt.subplots(figsize=(11, 4.2))
base = np.percentile(y, 5)
top = np.median(y) + 0.02 * (y.max() - y.min())
band = 0.09 * (top - base)
ax.plot(x, y, color="black", lw=0.7)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for j, p in enumerate(phases):
    pk_ = X.simulate_peaks(p["structure"], plot_range, X._emission_lines(WAVELENGTH)[0][0])
    pk_ = pk_[pk_.intensity >= 2]
    y0 = base - (j + 1) * band
    ax.vlines(pk_.two_theta, y0, y0 + 0.75 * band, color=colors[j % len(colors)], lw=1)
    ax.text(plot_range[0] + 0.5, y0 + 0.35 * band, f" {p['formula']}", color=colors[j % len(colors)], fontsize=8, va="center")
ax.set_ylim(base - (len(phases) + 0.3) * band, top)
ax.set_xlim(*plot_range); ax.set_xlabel(r"2$\theta$ / degrees"); ax.set_ylabel("counts (zoomed)")
ax.set_title("Zoom on weak peaks; ticks = reflections of each identified phase (Ka1, >= 2 %)")
plt.tight_layout(); plt.savefig(OUT / "phases_zoom.png", dpi=200); plt.show()